# Implementando Michigrad

**Conceptos clave**
* **Autograd (diferenciación automática):** cada operación (`+`, `*`, `tanh`, ...) guarda de qué valores salió y cómo calcular su derivada local. Con eso se arma un **grafo de cómputo**.
* **Backpropagation:** para saber cómo afecta cada peso a la loss $L$ recorremos el grafo **de atrás para adelante** aplicando la **regla de la cadena**: $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial out} \cdot \frac{\partial out}{\partial x}$.
* `grad` de cada nodo = $\frac{\partial L}{\partial \text{nodo}}$: cuánto cambia la loss si ese valor aumenta un poquito.
* Los gradientes se **acumulan** (`+=`) porque un mismo valor puede usarse en varias operaciones, y su efecto total es la suma.

In [ ]:
import torch
import numpy as np
from graphviz import Digraph

### La clase `Value`
Envuelve un número (`data`) y guarda:
* `grad`: la derivada de la loss respecto de este valor (arranca en 0).
* `_prev`: los valores de los que salió (sus hijos en el grafo).
* `_op`: la operación que lo generó.
* `_backward`: función que, dado `out.grad`, suma la contribución a los `grad` de sus hijos (regla de la cadena).

Derivadas locales usadas:
* Suma: $\frac{\partial (a+b)}{\partial a} = 1$, así que el gradiente pasa igual.
* Producto: $\frac{\partial (a \cdot b)}{\partial a} = b$.
* Potencia: $\frac{\partial a^n}{\partial a} = n \cdot a^{n-1}$.
* Exponencial: $\frac{\partial e^a}{\partial a} = e^a$.
* Tangente hiperbólica: $\frac{\partial \tanh a}{\partial a} = 1 - \tanh^2 a$.

`backward()` ordena el grafo **topológicamente** (cada nodo después de sus hijos), pone `grad = 1` en la salida ($\frac{\partial L}{\partial L} = 1$) y llama a `_backward` de cada nodo en orden inverso.

> **Corrección:** en `build_topo` decía `if v is not visited`, que compara si `v` es el mismo objeto que el set (siempre verdadero). Lo correcto es `if v not in visited`. Con el error, un nodo usado más de una vez se procesa varias veces y sus gradientes quedan mal.

In [ ]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def show_graph(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) #, node_attr={'rankdir': 'TB'})
    
    for n in nodes:
        dot.node(name=str(id(n)), label = "{%s | data %.4f | grad %.4f}" % (n.name, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot
    
class Value():
    def __init__(self, data, _children=(), _op="", name=""):
        self.data = data
        self.name = name
        self.grad = 0.0
        self._backward = lambda: None
        self._op = _op
        self._prev = set(_children)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self,other), "+")
        
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self,other), "*")

        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "solamente soportamos potencias de int/float"
        out = Value(self.data ** other, _children=(self, ), _op=f"**{other}")

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * other ** -1

    def exp(self):
        out = np.exp(self.data)
        out = Value(out, _children=(self,), _op="exp")
        
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward       
        return out

    def tanh(self):
        x = self.data
        t = (np.e ** (2*x) - 1)/(np.e ** (2*x) + 1)
        out = Value(t, _children=(self, ), _op="tanh")

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:   # (corregido: antes decía 'is not')
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.
        for node in reversed(topo):
            node._backward()

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __repr__(self):
        return f'Value(data={self.data}, grad={self.grad}, _op="{self._op}", name={self.name})'

### Una neurona a mano: $\hat{y} = \tanh(W_0 x_0 + W_1 x_1)$
Calculamos la salida de una neurona con dos entradas y una "loss" $L = y - \hat{y}$, y después hacemos backpropagation.

In [ ]:
W0 = Value(-.5, name='W₀')
x0 = Value(.5, name='x₀')
W1 = Value(-.3, name='W₁')
x1 = Value(2, name='x₁')
W0x0 = W0 * x0; W0x0.name = 'W0x0'
W1x1 = W1 * x1; W1x1.name = 'W1x1'
yhat = W0x0 + W1x1; yhat.name = 'ŷ'
yhat = yhat.tanh(); yhat.name = 'tanh(ŷ)'   # ŷ = tanh(W0x0 + W1x1)
y = Value(.8, name='y')
L = y - yhat; L.name = 'L'

In [ ]:
L.backward()
show_graph(L)   # cada nodo muestra su valor (data) y su gradiente (grad)

**Usando los gradientes:** el `grad` de cada peso dice cómo cambia `L` si el peso aumenta. Si queremos que `L` **baje**, movemos cada peso un poco en la dirección contraria a su gradiente. Probamos con pesos un poco distintos y vemos cómo cambian la salida y la loss:

In [ ]:
W0 = Value(-.4, name='W₀')
x0 = Value(.5, name='x₀')
W1 = Value(-.2, name='W₁')
x1 = Value(2, name='x₁')
W0x0 = W0 * x0; W0x0.name = 'W0x0'
W1x1 = W1 * x1; W1x1.name = 'W1x1'
yhat = W0x0 + W1x1; yhat.name = 'ŷ'
yhat = yhat.tanh(); yhat.name = 'tanh(ŷ)'
y = Value(.8, name='y')
L = y - yhat; L.name = 'L'

L.backward()
show_graph(L)

### Neurona, capa y MLP
* **Neurona:** $out = \tanh(\sum_i w_i x_i + b)$. Tiene un peso por entrada (`nin` pesos) y un sesgo `b`, todos inicializados al azar entre -1 y 1.
* **Capa (Layer):** `nout` neuronas que reciben **las mismas** entradas. Su salida es la lista de salidas de cada neurona.
* **MLP (Multi-Layer Perceptron):** capas en cadena: la salida de una capa es la entrada de la siguiente.

`parameters()` devuelve todos los pesos y sesgos, para poder actualizarlos en el entrenamiento.

In [ ]:
class Neuron:
    def __init__(self, nin):   # nin = número de entradas
        # np.random.uniform(a, b) devuelve un número entre a y b: una lista de nin Values entre -1 y 1
        self.W = [Value(np.random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(np.random.uniform(-1, 1))

    def __call__(self, x):
        # zip junta cada peso con su entrada: (W[0], x[0]), (W[1], x[1]), ...
        # (si una lista tiene más elementos, zip corta en la más corta)
        act = sum((wi * xi for wi, xi in zip(self.W, x)), self.b)   # w·x + b
        return act.tanh()

    def parameters(self):
        return self.W + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):   # nouts: lista con la cantidad de neuronas de cada capa
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
np.random.seed(42)
x = [1., 2., 3, 4, 5]
n = Neuron(len(x))   # una neurona con 5 entradas
show_graph(n(x))     # n(x) llama a __call__

### Entrenando un MLP
Un clasificador chico: 4 ejemplos con 3 entradas cada uno y su etiqueta (+1 o -1). Usamos un MLP de 3 entradas, dos capas ocultas de 4 neuronas y 1 salida.

Loss: **error cuadrático** $L = \sum (\hat{y} - y)^2$. En cada paso:
1. **Forward:** predicciones y loss.
2. **Zero grad:** ponemos los `grad` en 0 (si no, se acumulan de un paso al otro).
3. **Backward:** `loss.backward()`.
4. **Update:** `p.data -= lr * p.grad` para cada parámetro.

In [ ]:
np.random.seed(1)
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]   # salidas deseadas

model = MLP(3, [4, 4, 1])
lr = 0.05
for k in range(50):
    # forward
    ypred = [model(x) for x in xs]
    loss = sum((yp - yt)**2 for yp, yt in zip(ypred, ys))
    # zero grad
    for p in model.parameters():
        p.grad = 0.0
    # backward
    loss.backward()
    # update
    for p in model.parameters():
        p.data -= lr * p.grad
    if k % 10 == 0:
        print(k, round(loss.data, 4))

print('predicciones:', [round(yp.data, 3) for yp in ypred])
print('esperado:    ', ys)

## Conclusiones
* Hemos creado una pequeña librería para representar MLPs.
* La librería es suficientemente pontente para representar un clasificador sencillo.
* Es fácil de extender y la interfáz es en su mayor parte compatible con PyTorch.
* Lo visto debería alcanzar para comprender totalmente el código de Michigrad y Micrograd.

## Ejercicios
* Implementar el parámetro `bias` en `Neuron` y `Layer` para poder crear neuronas sin sesgo. (Fácil)
  * Pista: mirar la implementación de Michigrad.
* Implementar el modelo `xor` usando PyTorch. (Fácil si usaste Pytorch antes, Intermedio si no lo hiciste)
  * Pista: PyTorch no implementa un MLP. Se puede redefirnir la clase `MLP` usando `torch.Linear` en lugar de capas de neuronas (`Layer` y `Neuron`).
* Implementar las funciones de activación como capas. (Intermedio)
  * Pista: Reemplazar `Layer` por `Linear`, y definir una clase por cada función de activación. La capa `Linear` se comportará como una capa de neuronas y la capa de activación aplicará la función de activación a cada salida de todas las neuronas de la capa anterior.
* Implementar la clase Module que permita crear modelos como listas de modulos. Todos los módulos deben soportar la `__call__(self, x)` que hace la forward pass.  (Difícil)
  * Pista: Se tendría que poder crear un modelo como una lista de modulos, así:
   ```python
       model = Model([Linear(2, 4), Linear(4, 4, bias=False), Linear(4, 3), Tanh(3)])
       model(x)
   ```